In [1]:
#spark.stop()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("data_skew_homework").config("spark.sql.adaptive.enabled", "false").config("spark.sql.autoBroadcastJoinThreshold", "-1").config("spark.driver.memory", "15g").master("local[2]").getOrCreate()
#spark.conf.set("spark.sql.adaptive.enabled", "false")
#spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
# .config("spark.sql.shuffle.partitions", 100)
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/30 21:41:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
#spark.sparkContext.getConf().getAll()

### Read inputs

In [4]:
events_df = spark.read.json("events.json")
users_df = spark.read.csv("users.csv", header=True)

### Default join

In [5]:
from pyspark.sql.window import Window

events_users = events_df.join(users_df, on="user_id")
events_users.write.format("noop").mode("append").save()

In [6]:
events_users = events_users.withColumn("event_date_num", F.datediff(F.current_date(), F.to_date("event_timestamp")))
events_last_90_days = Window.partitionBy("user_id").orderBy("event_date_num").rangeBetween(0, 90)
events_users = events_users.withColumn("rolling_90d_events", F.count("*").over(events_last_90_days))
events_users.write.csv("events_with_rolling.csv", header=True, mode="overwrite")

### Salting

In [7]:
from pyspark.sql.window import Window

SALT_FACTOR = 50

# 1. Prepare data with salt for Join
events_salted = events_df.withColumn("salt", (F.rand() * SALT_FACTOR).cast("int")) \
                         .withColumn("salted_key", F.concat(F.col("user_id"), F.lit("_"), F.col("salt")))

users_salted = users_df.withColumn("salt", F.explode(F.array([F.lit(i) for i in range(SALT_FACTOR)]))) \
                       .withColumn("salted_key", F.concat(F.col("user_id"), F.lit("_"), F.col("salt")))

joined_salted = events_salted.join(users_salted.drop("user_id", "salt"), on="salted_key", how="inner")
#joined_salted.write.format("noop").mode("append").save()

In [8]:
# Add a numeric date representation for Window range evaluation
joined_salted = joined_salted.withColumn("event_date_num", F.datediff(F.current_date(), F.to_date("event_timestamp")))

# 2. Calculate the window within each salt
# Partitioning by salted_key distributes the heavy keys evenly across workers
window_local = Window.partitionBy("salted_key").orderBy("event_date_num").rangeBetween(0, 90)
df_local_aggr = joined_salted.withColumn("local_count", F.count("*").over(window_local))

# 3. Merging the salted results
# Group by original user_id and date to sum up metrics across all salts
df_final = df_local_aggr.groupBy("user_id", "event_date_num") \
                        .agg(F.sum("local_count").alias("rolling_90d_events"))

df_final.write.csv("events_with_rolling_simple.csv", header=True, mode="overwrite")

### Separate treatment

In [9]:
# 1. Find hot user_ids (top-N by event count)
top_users_rows = (
    events_df.groupBy("user_id")
    .count()
    .orderBy(F.col("count").desc())
    .limit(4)
    .collect()
)
print(top_users_rows)
hot_user_ids = [row["user_id"] for row in top_users_rows]

[Stage 12:=======================================================>(60 + 1) / 61]

[Row(user_id=0, count=16502803), Row(user_id=1, count=16498987), Row(user_id=2, count=16498210), Row(user_id=97855, count=18)]


In [10]:
# 2. Split events into hot and cold parts
events_hot = events_df.filter(F.col("user_id").isin(hot_user_ids))
events_cold = events_df.filter(~F.col("user_id").isin(hot_user_ids))

print("Hot events:", events_hot.count())
print("Cold events:", events_cold.count())

Hot events: 49500018


[Stage 16:======================================================> (59 + 2) / 61]

Cold events: 499982


In [11]:
# 3. Cold part: standard path, no tricks needed (many distinct keys, naturally balanced)
joined_cold = events_cold.join(users_df, on="user_id")
joined_cold = joined_cold.withColumn("event_date_num", F.datediff(F.current_date(), F.to_date("event_timestamp")))
window_cold = Window.partitionBy("user_id").orderBy("event_date_num").rangeBetween(0, 90)
joined_cold = joined_cold.withColumn("rolling_90d_events", F.count("*").over(window_cold))

In [12]:
# 4. Hot part: pre-aggregate to user+day level with salt (few keys, huge row counts)
SALT_FACTOR = 50
events_hot_salted = events_hot.withColumn("event_date_num", F.datediff(F.current_date(), F.to_date("event_timestamp"))) \
                              .withColumn("salt", (F.rand() * SALT_FACTOR).cast("int"))

partial_daily_counts = events_hot_salted.groupBy("user_id", "event_date_num", "salt").count()
daily_counts_hot = partial_daily_counts.groupBy("user_id", "event_date_num").agg(F.sum("count").alias("daily_count"))

In [13]:
# 5. Rolling window on the now-small daily aggregates (no salt needed, data is tiny per user)
window_hot_daily = Window.partitionBy("user_id").orderBy("event_date_num").rangeBetween(0, 90)
daily_counts_hot = daily_counts_hot.withColumn("rolling_90d_events", F.sum("daily_count").over(window_hot_daily))

In [14]:
# 6. Join rolling result back to original hot events + user info
events_hot_with_rolling = events_hot.withColumn("event_date_num", F.datediff(F.current_date(), F.to_date("event_timestamp"))) \
    .join(daily_counts_hot.select("user_id", "event_date_num", "rolling_90d_events"), on=["user_id", "event_date_num"], how="left") \
    .join(users_df, on="user_id")

In [15]:
# 7. Union hot and cold parts back together
final_df = joined_cold.unionByName(events_hot_with_rolling, allowMissingColumns=True)
final_df.write.format("noop").mode("append").save()

In [16]:
#events_users.groupBy("user_id").count().orderBy(F.col("count").desc()).show(20)

In [17]:
#events_users.write.format("noop").mode("append").save()